# L2 step 4 · The winning 3B on the benchmark's six tasks (GPU T4)

Task **L2** (roadmap), objective **O4**. Step 3 chose the winner by the rule fixed on 8 Nov; only the winner was to be scored here, once.

**Changed on 23 Nov, after the first full run**
- **The overflow:** the first run lost 327 of its 10,675 answers to a half-precision overflow. Each came out as 32 exclamation marks, the token picked when every score is NaN, and was scored wrong. It is L2 step 2b's overflow (19 Nov), at the same layer: the last block of the vision part, `visual.blocks.31.mlp.down_proj`. The training notebook carried step 2b's fix; the evaluation notebooks didn't.
- **The fix (cell 2b):** that layer runs in fp32, and the vision part under fp16 mixed precision, as in training. The first run is kept in `L2-step4-results/full-before-fix/`.
- **Run 4's choice isn't affected:** its 50 overflowed held-out questions were the same for all three 3Bs, since the adapters don't touch the vision part. And even all 50 can't turn 338 against 185.
- **The runner-up is scored too** (Abrar's comparison, 23 Nov), after the choice, so it can't change it. The two 3Bs answer at once, one per T4.
- **The smoke test** adds 12 questions that overflowed in the first run (2 per task).
- **Expectation, fixed before the rerun:**
  - no answer overflows;
  - every task's score for the winner is at least the first run's, less 0.5 points. Only wrong answers were lost, but the layer's higher precision may flip a few close answers.

**What it does:** the winning 3B (base `Qwen/Qwen2.5-VL-3B-Instruct` with its adapter merged in) answers the benchmark's 10,675 optical questions of the six tasks with L0 step 2's prompts and scorer, greedy, images at the 3B's training size. Since 23 Nov the runner-up answers them too. Answers are written as they come, so a stopped run continues.

**Reported beside accuracy**
- The new 7B at full size (L0 step 2) and the base 7B (the DisasterM3 paper), for scale.
- Building counting: count error against the true count, and the rule that never sees the image (66.98% on these questions), which accuracy on these options can't rule out.
- E5's clean set (its 186 copied questions dropped) and the counting options' pattern are checked on the PC from `per_question.csv` (`l2-step4b-clean-and-options-check.ipynb`).

**Expectation, fixed before the first run (8 Nov):** below the new 7B on most tasks, since the 3B is smaller; no number was predicted.

**How to run on Kaggle** (account 2)
1. Settings: GPU **T4 x2** (both are used, one 3B each), internet ON. Secret `HF_TOKEN` (AbrarAlam: the adapters are in a private repo).
2. Inputs: `redwannewaz/disasterm3-bench-mirror-v2`; Your Work → `l2-step3-heldout-select` (its `selection.json` names the winner).
3. Run All once with `SMOKE_TEST = True` (5 questions per task, plus the 12 that overflowed). Then `False` and Save & Run All. About 4–6 h (estimated; the first run took 4.1 h for one 3B alone). It stops cleanly at 11 h; to continue, attach that version's output and run again.
4. Outputs in `/kaggle/working/L2S4/`: `results.csv`, `per_question.csv` (the winner's `answer` and `correct`, and `answer_<3B>` and `correct_<3B>` for each), `README.md`, `config.json`, `answers.jsonl`.

In [1]:
# ── Cell 1: Environment: Kaggle's torch and transformers, current peft (as L2 step 2), and no torchao ──
# L0's merge (26 Oct) stopped on "Found an incompatible version of torchao": peft checks torchao when it puts an adapter
# on an fp16 model, as this notebook does. Nothing here uses torchao.
import importlib.metadata as _md
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft"], check=True)

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

assert importlib.util.find_spec("torchao") is None, "torchao is still installed: restart the session and run again"
assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16.1 on 26 Oct)"
import torch
VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}; torchao: not installed")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 1, "Pick GPU T4 x2 (one is used)"


✓ {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'peft': '0.22.0'}, python 3.13.16; torchao: not installed
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration: the winner, the benchmark ─────────────────────────
import ast, glob, json, math, os, re, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoProcessor

BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_REPO = "AbrarAlam/disasterm3-qwen25vl3b-l2"
BATCH, MAX_NEW_TOKENS = 8, 32
SMOKE_TEST = True             # True = 5 questions per task
TIME_LIMIT_HOURS = 11.0
OUT_DIR = Path("/kaggle/working/L2S4")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS_PATH = OUT_DIR / "answers.jsonl"
CUDA = torch.cuda.is_available()
EXPECTED_QUESTIONS = {"DTR": 353, "BBR": 2006, "BDC": 3583, "DRE": 1868, "DSR": 2007, "ORR": 858}
REF_NEW_7B_FULL = {"DTR": 76.77, "BBR": 19.99, "BDC": 30.31, "DRE": 33.03, "DSR": 40.46, "ORR": 34.62}   # L0 step 2
REF_BASE_7B = {"DTR": 66.6, "BBR": 4.7, "BDC": 34.2, "DRE": 29.3, "DSR": 28.3, "ORR": 23.9}              # DisasterM3 Table 2

import glob as _g, os as _o
for _f in _g.glob("/kaggle/input/**/hf_write_token.txt", recursive=True):   # runs started from the PC have no secrets
    _o.environ["HF_TOKEN"] = open(_f, encoding="utf-8").read().strip()

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})
# ── The benchmark's prompts and scorer, and the 3Bs (the same code in steps 3 and 4) ──
# The four prompts, copied from L0 step 2 (which copied them from the DisasterM3 authors' runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)
# (code, subset name in the 499A scorer, task name in the benchmark, task name in train_release.json)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", "Disaster Type Recognition"),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", "Disaster Bearing Bodies Recognition"),
         ("BDC", "building_damage_counting", "Building Damage Counting", "Building Damage Counting"),
         ("DRE", "road_damage_counting", "Road Damage Counting", "Road Damage Counting"),
         ("DSR", "landuse", "Disaster Scene Recognition", "Disaster Scene Recognition"),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", "relational reasoning")]
COUNTING = ("BDC", "DRE")

def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def prompt_text(e, subset):
    """(the benchmark prompt, the image keys in order), as L0 step 2's build() makes them."""
    p = e.get("prompts", "")
    p = p[0] if isinstance(p, list) else p
    if subset == "landuse":
        return PROMPTS["landuse"].format(prompt=p, options_str=e["options_str"]), ["pre_image_path"]
    if subset == "relational_reasoning_qa":           # its entries spell the options' field option_str
        return PROMPTS[subset].format(prompt=p, options_str=e["option_str"]), ["image_path"]
    key = "bearing_body" if subset == "bearing_body" else "single_choice"
    return PROMPTS[key].format(prompt=p, options_str=e["options_str"]), ["pre_image_path", "post_image_path"]

def content_of(text, n_images):
    """L0 step 2's layout: single-image prompts put the image first; two-image prompts fill each '<image>' gap."""
    parts = text.split("<image>")
    if len(parts) == 1:
        return [{"type": "image"} for _ in range(n_images)] + [{"type": "text", "text": text}]
    content = []
    for n, part in enumerate(parts):
        if part:
            content.append({"type": "text", "text": part})
        if n < len(parts) - 1:
            content.append({"type": "image"})
    return content

def option_values(e):
    """The five option values of a counting question, as numbers ('5.22%' -> 5.22)."""
    o = e["options_list"]
    o = ast.literal_eval(o) if isinstance(o, str) else o
    return [float(str(x).rstrip("%")) for x in o]

def no_image_rule(values):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest (benchmark check, 7 Nov)."""
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

def chosen_value(answer, values):
    """The value of the one option an answer names; None if it names no option or several."""
    got = letters(answer)
    if len(got) != 1 or next(iter(got)) not in "ABCDE"[:len(values)]:
        return None
    return values["ABCDE".index(next(iter(got)))]

assert letters("The answer is D.") == {"D"} and no_image_rule([231, 323, 277, 139, 185]) == "A"
assert chosen_value("B", [1, 2, 3, 4, 5]) == 2 and chosen_value("B, C", [1, 2, 3, 4, 5]) is None
assert content_of("x<image>y<image>z", 2)[1] == {"type": "image"} and content_of("q", 1)[0] == {"type": "image"}

def load_3b(which):
    """The base 3B in fp16 on one T4; for 'b' or 'c', its finished adapter merged in."""
    from peft import PeftModel
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float16 if CUDA else torch.float32,
                                                               device_map={"": 0} if CUDA else None, attn_implementation="sdpa")
    if which != "base":
        local = snapshot_download(ADAPTER_REPO, allow_patterns=[f"{which}/final/*"], token=hf_token())
        model = PeftModel.from_pretrained(model, str(Path(local) / which / "final")).merge_and_unload()
    return model.eval()

@torch.inference_mode()
def answer_batch(model, items):
    """items: (prompt text, image paths) -> greedy answers, every sampling setting given, images at the 3B's cap."""
    texts, images = [], []
    for text, paths in items:
        imgs = [sized(Image.open(p).convert("RGB")) for p in paths]
        texts.append(processor.apply_chat_template([{"role": "user", "content": content_of(text, len(imgs))}], tokenize=False,
                                                   add_generation_prompt=True))
        images += imgs
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0)
    return [s.strip() for s in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

def answer_safe(model, items):
    """Halves the batch on out-of-memory or an unreadable image; a question that still fails alone gets '' (scored wrong,
    as L0 step 2 scored its unreadable images)."""
    try:
        return answer_batch(model, items)
    except (torch.cuda.OutOfMemoryError, OSError):
        torch.cuda.empty_cache()
        if len(items) == 1:
            return [""]
        half = len(items) // 2
        return answer_safe(model, items[:half]) + answer_safe(model, items[half:])

selection = json.loads(one("L2S3/selection.json", "attach Your Work -> l2-step3-heldout-select").read_text(encoding="utf-8"))
WINNER = selection["winner"]
assert WINNER in ("b", "c"), "step 3's output holds no winner: was it a smoke test?"
BENCH_JSON = one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2")
processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.tokenizer.padding_side = "left"
print(f"✓ L2 step 4{'  [SMOKE TEST]' if SMOKE_TEST else ''}: the winner is ({WINNER}): {selection['reason']}")


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✓ L2 step 4  [SMOKE TEST]: the winner is (c): primary accuracy 35.95% (b) against 45.12% (c), exact McNemar p = 0.0000


In [3]:
# ── Cell 2b (added 23 Nov): the overflow fix, and the runner-up scored too, one 3B per T4 ──
# The first full run lost 327 answers to a half-precision overflow at the vision part's last block, the layer L2 step 2b
# found in training (19 Nov). As in training, that layer runs in fp32 and the vision part under fp16 mixed precision
# (autocast), which hands its output back to the language model in fp16. These definitions replace cell 2's load_3b,
# answer_batch and answer_safe.
import threading

FP32_LAYER = "visual.blocks.31.mlp.down_proj"    # L2 step 2b's layer
DEVICE = "cuda" if CUDA else "cpu"
ALSO_RUNNER_UP = True        # Abrar's comparison (23 Nov): the runner-up on the benchmark too, after the choice
OVERFLOWED_BEFORE = ["disaster_type_331", "disaster_type_335", "bearing_body_1641", "bearing_body_1645",
                     "building_damage_counting_1904", "building_damage_counting_1908", "road_damage_counting_1578",
                     "road_damage_counting_1582", "landuse_1451", "landuse_1454", "relational_reasoning_qa_568",
                     "relational_reasoning_qa_569"]   # overflowed in the first full run (2 per task): the smoke test adds them
OVERFLOWED = re.compile(r"!+")                       # what an overflowed answer looks like


def fp32_islands(model, names):
    """L2 step 2's function: these modules run in fp32 with autocast off (their inputs, their fp16 weights and their
    4-bit layers' compute)."""
    mods = dict(model.named_modules())
    for n in names:
        m = mods[n]

        def forward(*args, __orig=m.forward, **kwargs):
            with torch.autocast(DEVICE, enabled=False):
                args = tuple(a.float() if torch.is_tensor(a) and a.is_floating_point() else a for a in args)
                kwargs = {k: v.float() if torch.is_tensor(v) and v.is_floating_point() else v for k, v in kwargs.items()}
                return __orig(*args, **kwargs)

        m.forward = forward
        for sub in m.modules():
            if hasattr(sub, "compute_dtype"):
                sub.compute_dtype = torch.float32
            for p in sub.parameters(recurse=False):
                if p.is_floating_point() and p.dtype != torch.float32:
                    p.data = p.data.float()


def with_fix(model):
    """Step 2b's layer in fp32, inside a vision part run under fp16 autocast, as in training (tested on the PC on a tiny
    random Qwen2.5-VL, 23 Nov: the layer computes in fp32, the image features reach the language model in fp16)."""
    islands = [n for n, _ in model.named_modules() if n.endswith(FP32_LAYER)]
    towers = [m for n, m in model.named_modules() if n.endswith("visual") and hasattr(m, "blocks")]
    assert len(islands) == 1 and len(towers) == 1, f"{len(islands)} of step 2b's layer, {len(towers)} vision parts"
    fp32_islands(model, islands)
    orig = towers[0].forward

    def forward(*args, **kwargs):
        with torch.autocast(DEVICE, dtype=torch.float16):
            return orig(*args, **kwargs)

    towers[0].forward = forward
    return model


def load_3b(which, gpu=0):
    """The base 3B in fp16 on one T4 (GPU `gpu`); for 'b' or 'c', its finished adapter merged in; then the fix."""
    from peft import PeftModel
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float16 if CUDA else torch.float32,
                                                               device_map={"": gpu} if CUDA else None, attn_implementation="sdpa")
    if which != "base":
        local = snapshot_download(ADAPTER_REPO, allow_patterns=[f"{which}/final/*"], token=hf_token())
        model = PeftModel.from_pretrained(model, str(Path(local) / which / "final")).merge_and_unload()
    return with_fix(model).eval()


@torch.inference_mode()
def answer_batch(model, items, proc):
    """items: (prompt text, image paths) -> greedy answers, every sampling setting given, images at the 3B's cap. `proc`
    is this 3B's own processor (one per thread)."""
    texts, images = [], []
    for text, paths in items:
        imgs = [sized(Image.open(p).convert("RGB")) for p in paths]
        texts.append(proc.apply_chat_template([{"role": "user", "content": content_of(text, len(imgs))}], tokenize=False,
                                              add_generation_prompt=True))
        images += imgs
    inputs = proc(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0)
    return [s.strip() for s in proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]


def answer_safe(model, items, proc):
    """Halves the batch on out-of-memory or an unreadable image; a question that still fails alone gets '' (scored wrong,
    as L0 step 2 scored its unreadable images)."""
    try:
        return answer_batch(model, items, proc)
    except (torch.cuda.OutOfMemoryError, OSError):
        torch.cuda.empty_cache()
        if len(items) == 1:
            return [""]
        half = len(items) // 2
        return answer_safe(model, items[:half], proc) + answer_safe(model, items[half:], proc)


RUNNER_UP = "b" if WINNER == "c" else "c"
MODELS = [WINNER] + ([RUNNER_UP] if ALSO_RUNNER_UP else [])
assert not CUDA or torch.cuda.device_count() >= len(MODELS), "Pick GPU T4 x2: each 3B answers on its own GPU"
print(f"✓ The fix: {FP32_LAYER} in fp32, the vision part under fp16 autocast; answering {MODELS}, one per GPU")

✓ The fix: visual.blocks.31.mlp.down_proj in fp32, the vision part under fp16 autocast; answering ['c', 'b'], one per GPU


In [4]:
# ── Cell 3: The benchmark's optical questions of the six tasks (as L0 step 2 asked them) ──
with open(BENCH_JSON, encoding="utf-8") as f:
    bench = json.load(f)
bench_index = {}                # every benchmark image by file name (L0 step 2's rule: mirrors nest them differently)
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

def sensor(e):
    """H1 V2's rule for the after-image's sensor: most entries have post_image_type, ORR entries image_type."""
    return e.get("post_image_type") or e.get("image_type") or "Optical"

rows = []
for code, subset, bench_task, _ in TASKS:      # ids as in H1 V2, L0 step 2 and E5: the index among the task's entries
    entries = [(f"{subset}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == bench_task)]
    entries = [(qid, e) for qid, e in entries if sensor(e) == "Optical"]
    assert len(entries) == EXPECTED_QUESTIONS[code], f"{code}: {len(entries)} optical questions"
    for qid, e in entries:
        text, keys = prompt_text(e, subset)
        row = {"id": qid, "task": code, "text": text, "images": [bench_index.get(name_of(e[k])) for k in keys], "truth": e["ground_truth_option"]}
        if code in COUNTING:
            values = option_values(e)
            right = sorted(letters(e["ground_truth_option"]))[0]
            row.update(values=values, truth_value=values["ABCDE".index(right)], rule_correct=no_image_rule(values) == right)
        rows.append(row)
Q = pd.DataFrame(rows)
Q["image_missing"] = Q["images"].map(lambda ps: any(p is None for p in ps))
assert Q["image_missing"].sum() <= 0.01 * len(Q), f"{Q['image_missing'].sum()} questions have an image missing: attach the bench mirror"
assert set(OVERFLOWED_BEFORE) <= set(Q["id"]), "a question that overflowed before is missing"
print(f"✓ {len(Q):,} optical questions { {k: int(v) for k, v in Q['task'].value_counts().items()} }; "
      f"{int(Q['image_missing'].sum())} with an image missing "
      "(counted wrong)")
if SMOKE_TEST:                  # 5 per task, and (23 Nov) the 12 that overflowed in the first full run
    Q = Q[Q.index.isin(Q.groupby("task").head(5).index) | Q["id"].isin(OVERFLOWED_BEFORE)].reset_index(drop=True)

✓ 10,675 optical questions {'BDC': 3583, 'DSR': 2007, 'BBR': 2006, 'DRE': 1868, 'ORR': 858, 'DTR': 353}; 0 with an image missing (counted wrong)


In [5]:
# ── Cell 4: The 3Bs answer at once, one per T4, written as they go (resumable) ────
from tqdm.auto import tqdm

previous = sorted(glob.glob("/kaggle/input/**/L2S4/answers.jsonl", recursive=True))
if previous and not ANSWERS_PATH.exists():
    shutil.copy(previous[0], ANSWERS_PATH)
done = {m: {} for m in MODELS}
if ANSWERS_PATH.exists():
    with open(ANSWERS_PATH, encoding="utf-8") as f:
        for r in map(json.loads, f):
            if r.get("model") in done and r.get("fix") == FP32_LAYER:      # only answers made with the fix (23 Nov)
                done[r["model"]][r["id"]] = r["answer"]
print({m: f"{len(done[m]):,} already answered" for m in MODELS})
models, procs = {}, {}
for gpu, m in enumerate(MODELS):                 # loaded one after the other, each on its own GPU
    models[m] = load_3b(m, gpu=gpu)
    procs[m] = AutoProcessor.from_pretrained(BASE_MODEL)
    procs[m].tokenizer.padding_side = "left"
deadline, t0 = time.time() + TIME_LIMIT_HOURS * 3600, time.time()
lock, stopped, seconds, errors = threading.Lock(), {}, {}, {}


def work(m, gpu, fh):
    """One 3B on its own GPU: answers its remaining questions in batches; the shared file is written under a lock."""
    try:
        if CUDA:
            torch.cuda.set_device(gpu)           # so out-of-memory recovery empties this GPU's cache
        todo = Q[~Q["id"].isin(list(done[m])) & ~Q["image_missing"]]
        start_time = time.time()
        for start in tqdm(range(0, len(todo), BATCH), desc=f"3B ({m})", position=gpu):
            if time.time() > deadline:
                stopped[m] = True
                break
            batch = todo.iloc[start:start + BATCH]
            answers = answer_safe(models[m], list(zip(batch["text"], batch["images"])), procs[m])
            with lock:
                for qid, answer in zip(batch["id"], answers):
                    fh.write(json.dumps({"id": qid, "model": m, "answer": answer, "fix": FP32_LAYER}) + "\n")
                    done[m][qid] = answer
                fh.flush()
        seconds[m] = round(time.time() - start_time, 1)
    except Exception as e:                       # recorded here, raised once both threads have ended
        errors[m] = repr(e)


with open(ANSWERS_PATH, "a", encoding="utf-8") as fh:
    threads = [threading.Thread(target=work, args=(m, gpu, fh)) for gpu, m in enumerate(MODELS)]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
assert not errors, f"a 3B stopped: {errors}"
RUN_HOURS = (time.time() - t0) / 3600
print(f"{'Time limit reached: stopped cleanly' if stopped else 'All answered'} after {RUN_HOURS:.2f} h; seconds per 3B: {seconds}")

{'c': '0 already answered', 'b': '0 already answered'}


model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

3B (b):   0%|          | 0/6 [00:00<?, ?it/s]

3B (c):   0%|          | 0/6 [00:00<?, ?it/s]

All answered after 0.02 h; seconds per 3B: {'c': 67.6, 'b': 71.2}


In [6]:
# ── Cell 5: Scores against the references, count error, the overflow check, and the output files ──
for m in MODELS:
    Q[f"answer_{m}"] = Q["id"].map(done[m])
    Q[f"correct_{m}"] = [isinstance(a, str) and letters(a) == letters(t) for a, t in zip(Q[f"answer_{m}"], Q["truth"])]
Q["answer"], Q["correct"] = Q[f"answer_{WINNER}"], Q[f"correct_{WINNER}"]     # the winner, under the first run's names
complete = all(bool((Q[f"answer_{m}"].notna() | Q["image_missing"]).all()) for m in MODELS)
overflowed = {m: int(sum(bool(OVERFLOWED.fullmatch(str(a).strip())) for a in done[m].values())) for m in MODELS}
table = []
for code, _, _, _ in TASKS:
    g = Q[Q["task"] == code]
    row = {"task": code, "questions": len(g)}
    row.update({f"3B_{m}_%": round(100 * g[f"correct_{m}"].mean(), 2) if len(g) else float("nan") for m in MODELS})
    row.update({"new_7B_full_%": REF_NEW_7B_FULL[code], "base_7B_paper_%": REF_BASE_7B[code]})
    table.append(row)
results = pd.DataFrame(table)


def counting_of(m):
    """Building counting for one 3B: accuracy, the chosen option's count error, and the rule that never sees the image."""
    bdc = Q[Q["task"] == "BDC"].copy()
    bdc["value"] = [chosen_value(a, v) if isinstance(a, str) else None for a, v in zip(bdc[f"answer_{m}"], bdc["values"])]
    err = (bdc["value"].astype(float) - bdc["truth_value"]).abs() / bdc["truth_value"].clip(lower=1)
    return {"accuracy_%": round(100 * float(bdc[f"correct_{m}"].mean()), 2),
            "median_count_error_%": round(100 * float(err.median()), 1) if err.notna().any() else None,
            "answered_with_one_option_%": round(100 * float(bdc["value"].notna().mean()), 1),
            "no_image_rule_%": round(100 * float(bdc["rule_correct"].mean()), 2)}


counting = {m: counting_of(m) for m in MODELS}
print(results.to_string(index=False))
print(f"\nBuilding counting: {counting}\nOverflowed answers (only exclamation marks): {overflowed}")
if SMOKE_TEST:
    print("The 12 that overflowed in the first full run, now:", {m: [done[m].get(i) for i in OVERFLOWED_BEFORE] for m in MODELS})


def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


Q.drop(columns=["text"]).to_csv(OUT_DIR / "per_question.csv", index=False)
results.to_csv(OUT_DIR / "results.csv", index=False)
config = {"task": "L2 step 4: the winning 3B (and since 23 Nov the runner-up) on the benchmark", "smoke_test": SMOKE_TEST,
          "winner": WINNER, "models": MODELS, "selection": selection, "base_model": BASE_MODEL,
          "adapters": {m: f"{ADAPTER_REPO}/{m}/final" for m in MODELS}, "max_pixels": MAX_PIXELS, "decoding": "greedy",
          "fix": f"{FP32_LAYER} in fp32; the vision part under fp16 autocast (23 Nov)", "overflowed_answers": overflowed,
          "complete": complete, "hours_this_session": round(RUN_HOURS, 2), "seconds_per_3b": seconds,
          "building_counting": counting, "versions": VERSIONS}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")
names = {WINNER: "the winner", RUNNER_UP: "the runner-up"}
(OUT_DIR / "README.md").write_text("\n".join([
    f"# L2 step 4: the winning 3B, option ({WINNER})" + (f", and the runner-up ({RUNNER_UP})" if ALSO_RUNNER_UP else "")
    + ", on the benchmark" + (" (SMOKE TEST)" if SMOKE_TEST else ""), "",
    f"`{BASE_MODEL}` with each adapter (`{ADAPTER_REPO}/<3B>/final`) merged in, on the benchmark's optical questions of the six "
    f"tasks, with L0 step 2's prompts and scorer, greedy, images at most {MAX_PIXELS:,} pixels. Step 3's reason for the winner: "
    f"{selection['reason']}. The runner-up was scored after the choice, so it couldn't change it.", "",
    f"**The overflow fix (23 Nov):** `{FP32_LAYER}` runs in fp32 and the vision part under fp16 autocast, as in training. "
    f"Overflowed answers: {', '.join(f'{names[m]} {overflowed[m]}' for m in MODELS)} (the first full run, without the fix: 327 for (c)).", "",
    "## Accuracy (%)", "", as_markdown(results), "",
    "The new 7B's numbers are at full size (1036 px); the 3Bs read their training size. The base 7B's are the DisasterM3 paper's (Table 2).", "",
    "## Building counting, beside accuracy", "",
    *[f"- {names[m]} ({m}): accuracy {counting[m]['accuracy_%']}%; median count error {counting[m]['median_count_error_%']}% "
      f"(the chosen option's value against the true count), on the {counting[m]['answered_with_one_option_%']}% answered with one option."
      for m in MODELS],
    f"- The rule that never sees the image: {counting[WINNER]['no_image_rule_%']}%. Accuracy on these options can't show that a "
    "model counts well; `l2-step4b-clean-and-options-check.ipynb` tests it on the questions whose options don't give the answer away.", "",
    "## Files", "", "- `results.csv`, `per_question.csv`, `answers.jsonl` (every answer), `config.json`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUT_DIR}: results.csv, per_question.csv, answers.jsonl, README.md, config.json"
      + ("" if complete else " (not complete: attach this output and run again)"))

task  questions  3B_c_%  3B_b_%  new_7B_full_%  base_7B_paper_%
 DTR          7   28.57    0.00          76.77             66.6
 BBR          7   14.29   28.57          19.99              4.7
 BDC          7   28.57   28.57          30.31             34.2
 DRE          7   42.86   14.29          33.03             29.3
 DSR          7   42.86   57.14          40.46             28.3
 ORR          7   42.86    0.00          34.62             23.9

Building counting: {'c': {'accuracy_%': 28.57, 'median_count_error_%': 17.2, 'answered_with_one_option_%': 100.0, 'no_image_rule_%': 85.71}, 'b': {'accuracy_%': 28.57, 'median_count_error_%': 18.5, 'answered_with_one_option_%': 100.0, 'no_image_rule_%': 85.71}}
Overflowed answers (only exclamation marks): {'c': 0, 'b': 0}
The 12 that overflowed in the first full run, now: {'c': ['E', 'D', 'G', 'B, E', 'E', 'B', 'B', 'B', 'E, G', 'F, D, B', 'E', 'D'], 'b': ['E', 'C', 'A, G', 'A, B, E', 'A', 'A', 'A', 'A', 'E, G', 'F, D, B', 'E', 'A']}
✓ Wrote /ka